# Train/Test Split Visualization — Random vs Kennard-Stone

Visualizes the distribution of training and testing sets:
1. **Random selection** (stratified, seed=42)
2. **Kennard-Stone algorithm** (maximin distance on ECFP4)

Both 80/20 split on 3,290 molecules with exact pchembl_value.

## 1. Mount Drive & Load Data

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

!cp "/content/drive/My Drive/data.zip" /content/data.zip
!unzip -qo /content/data.zip -d /content/
print('Data ready.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

BASE = "/content/data"

df_full = pd.read_csv(f"{BASE}/processed/aromatase_bioactivity_clean.csv")
mask = (df_full["standard_relation"] == "=") & df_full["pchembl_value"].notna()
df = df_full[mask].reset_index(drop=True)
print(f"Dataset: {len(df)} molecules")

random_train = pd.read_csv(f"{BASE}/splits/random_train.csv")
random_test = pd.read_csv(f"{BASE}/splits/random_test.csv")
ks_train = pd.read_csv(f"{BASE}/splits/kennard_stone_train.csv")
ks_test = pd.read_csv(f"{BASE}/splits/kennard_stone_test.csv")

df["random_set"] = "Unassigned"
df.loc[df["molecule_chembl_id"].isin(random_train["molecule_chembl_id"]), "random_set"] = "Train"
df.loc[df["molecule_chembl_id"].isin(random_test["molecule_chembl_id"]), "random_set"] = "Test"
df["ks_set"] = "Unassigned"
df.loc[df["molecule_chembl_id"].isin(ks_train["molecule_chembl_id"]), "ks_set"] = "Train"
df.loc[df["molecule_chembl_id"].isin(ks_test["molecule_chembl_id"]), "ks_set"] = "Test"

print(f"Random: Train={random_train.shape[0]}, Test={random_test.shape[0]}")
print(f"KS:     Train={ks_train.shape[0]}, Test={ks_test.shape[0]}")

fp_full = pd.read_csv(f"{BASE}/fingerprints_filtered/fingerprints_ecfp4.csv")
fp = fp_full[mask.values].reset_index(drop=True)
X_ecfp4 = fp.iloc[:, 1:].values.astype(np.float32)
X_ecfp4 = np.nan_to_num(X_ecfp4, nan=0.0)
print(f"ECFP4: {X_ecfp4.shape[1]} bits")


## 2. pchembl_value Distribution (Train vs Test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [("Train", "#3498db"), ("Test", "#e74c3c")]:
    sub = df[df["random_set"] == label]["pchembl_value"]
    axes[0].hist(sub, bins=40, alpha=0.6, label=f"{label} (n={len(sub)}, mean={sub.mean():.2f})",
                 color=color, density=True)
    sub.plot.kde(ax=axes[0], color=color, lw=2)
axes[0].set_xlabel("pchembl_value")
axes[0].set_ylabel("Density")
axes[0].set_title("Random Split")
axes[0].legend()

for label, color in [("Train", "#3498db"), ("Test", "#e74c3c")]:
    sub = df[df["ks_set"] == label]["pchembl_value"]
    axes[1].hist(sub, bins=40, alpha=0.6, label=f"{label} (n={len(sub)}, mean={sub.mean():.2f})",
                 color=color, density=True)
    sub.plot.kde(ax=axes[1], color=color, lw=2)
axes[1].set_xlabel("pchembl_value")
axes[1].set_ylabel("Density")
axes[1].set_title("Kennard-Stone Split")
axes[1].legend()

plt.suptitle("pchembl_value Distribution: Train vs Test", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/split_pchembl_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Box Plot Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df[df["random_set"] != "Unassigned"], x="random_set", y="pchembl_value",
            palette={"Train": "#3498db", "Test": "#e74c3c"}, ax=axes[0], width=0.5)
axes[0].set_title("Random Split")
axes[0].set_xlabel("")

sns.boxplot(data=df[df["ks_set"] != "Unassigned"], x="ks_set", y="pchembl_value",
            palette={"Train": "#3498db", "Test": "#e74c3c"}, ax=axes[1], width=0.5)
axes[1].set_title("Kennard-Stone Split")
axes[1].set_xlabel("")

plt.suptitle("pchembl_value: Train vs Test", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/split_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Activity Class Balance

In [ ]:
def classify(val):
    if val > 7: return "active"
    elif val < 6: return "inactive"
    else: return "intermediate"

df["activity_class"] = df["pchembl_value"].apply(classify)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (split_col, split_name) in enumerate([("random_set", "Random"), ("ks_set", "Kennard-Stone")]):
    for j, subset in enumerate(["Train", "Test"]):
        ax = axes[i, j]
        sub = df[df[split_col] == subset]
        counts = sub["activity_class"].value_counts()
        colors = {"active": "#2ecc71", "intermediate": "#f39c12", "inactive": "#e74c3c"}
        order = ["active", "intermediate", "inactive"]
        ax.bar(order, [counts.get(c, 0) for c in order],
               color=[colors[c] for c in order], edgecolor="black", linewidth=0.5)
        for k, c in enumerate(order):
            ax.text(k, counts.get(c, 0) + 5, str(counts.get(c, 0)), ha="center", fontweight="bold")
        ax.set_title(f"{split_name} - {subset} (n={len(sub)})")
        ax.set_ylabel("Count")

plt.suptitle("Activity Class Balance", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/content/split_activity_balance.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. PCA Chemical Space (Train vs Test)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_ecfp4)
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for label, color, alpha, s in [("Train", "#3498db", 0.3, 8), ("Test", "#e74c3c", 0.7, 15)]:
    m = df["random_set"] == label
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=color, label=label, alpha=alpha, s=s, edgecolors="none")
axes[0].set_xlabel(f"PC1 ({var_exp[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({var_exp[1]*100:.1f}%)")
axes[0].set_title("Random Split - PCA")
axes[0].legend(markerscale=2)

for label, color, alpha, s in [("Train", "#3498db", 0.3, 8), ("Test", "#e74c3c", 0.7, 15)]:
    m = df["ks_set"] == label
    axes[1].scatter(X_pca[m, 0], X_pca[m, 1], c=color, label=label, alpha=alpha, s=s, edgecolors="none")
axes[1].set_xlabel(f"PC1 ({var_exp[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({var_exp[1]*100:.1f}%)")
axes[1].set_title("Kennard-Stone Split - PCA")
axes[1].legend(markerscale=2)

plt.suptitle("Chemical Space Coverage (ECFP4 PCA)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/split_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("KS: training covers periphery, test is interior.")
print("Random: train and test overlap uniformly.")


## 6. t-SNE Chemical Space

In [ ]:
print("Running t-SNE (~1-2 min)...")
pca50 = PCA(n_components=50, random_state=42)
X_pca50 = pca50.fit_transform(X_ecfp4)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, init="pca")
X_tsne = tsne.fit_transform(X_pca50)
print("Done.")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for label, color, alpha, s in [("Train", "#3498db", 0.3, 8), ("Test", "#e74c3c", 0.7, 15)]:
    m = df["random_set"] == label
    axes[0].scatter(X_tsne[m, 0], X_tsne[m, 1], c=color, label=label, alpha=alpha, s=s, edgecolors="none")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
axes[0].set_title("Random Split - t-SNE")
axes[0].legend(markerscale=2)

for label, color, alpha, s in [("Train", "#3498db", 0.3, 8), ("Test", "#e74c3c", 0.7, 15)]:
    m = df["ks_set"] == label
    axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=color, label=label, alpha=alpha, s=s, edgecolors="none")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].set_title("Kennard-Stone Split - t-SNE")
axes[1].legend(markerscale=2)

plt.suptitle("Chemical Space (ECFP4 t-SNE)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/split_tsne.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Molecular Property Distributions

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
RDLogger.logger().setLevel(RDLogger.ERROR)

def calc_desc(smi):
    mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
    if mol is None: return [np.nan]*4
    return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.NumHAcceptors(mol), Descriptors.NumHDonors(mol)]

print("Computing descriptors...")
desc = df["canonical_smiles"].apply(calc_desc)
desc_df = pd.DataFrame(desc.tolist(), columns=["MW", "LogP", "HBA", "HBD"], index=df.index)
df = pd.concat([df, desc_df], axis=1)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
props = ["MW", "LogP", "HBA", "HBD"]
for row, (split_col, split_name) in enumerate([("random_set", "Random"), ("ks_set", "Kennard-Stone")]):
    for col_i, prop in enumerate(props):
        ax = axes[row, col_i]
        for label, color in [("Train", "#3498db"), ("Test", "#e74c3c")]:
            sub = df[df[split_col] == label][prop].dropna()
            ax.hist(sub, bins=30, alpha=0.5, color=color, label=label, density=True)
        ax.set_title(f"{split_name} - {prop}")
        ax.set_xlabel(prop)
        if col_i == 0: ax.set_ylabel("Density")
        if row == 0 and col_i == 0: ax.legend()

plt.suptitle("Molecular Properties: Train vs Test", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/content/split_properties.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Summary Statistics

In [ ]:
print("=" * 80)
print(f"{'SPLIT COMPARISON SUMMARY':^80}")
print("=" * 80)

for split_col, split_name in [("random_set", "Random"), ("ks_set", "Kennard-Stone")]:
    print(f"\n--- {split_name} Split ---")
    train = df[df[split_col] == "Train"]["pchembl_value"]
    test = df[df[split_col] == "Test"]["pchembl_value"]
    print(f"  {'Metric':<15} {'Train':<12} {'Test':<12} {'Diff':<8}")
    print(f"  {'-'*50}")
    for name, tv, ev in [("n", len(train), len(test)),
                          ("Mean", train.mean(), test.mean()),
                          ("Std", train.std(), test.std()),
                          ("Median", train.median(), test.median()),
                          ("Min", train.min(), test.min()),
                          ("Max", train.max(), test.max())]:
        if name == "n":
            print(f"  {name:<15} {tv:<12} {ev:<12}")
        else:
            print(f"  {name:<15} {tv:<12.4f} {ev:<12.4f} {abs(tv-ev):<8.4f}")

print("\nKey observations:")
print("  Random: distributions well-matched (stratified by design)")
print("  KS: training spans chemical space extremes, test is interior/clustered")


## 9. Save to Drive

In [ ]:
import shutil
from google.colab import files

figures = ["split_pchembl_distribution.png", "split_boxplot.png",
           "split_activity_balance.png", "split_pca.png",
           "split_tsne.png", "split_properties.png"]
try:
    for f in figures:
        shutil.copy(f"/content/{f}", f"/content/drive/My Drive/{f}")
    print("Saved to Google Drive.")
except Exception as e:
    print(f"Drive save: {e}")

for f in figures:
    if os.path.exists(f"/content/{f}"):
        files.download(f"/content/{f}")
print("Done.")
